In [7]:
import pulp as lp

In [8]:
nama_alat = {
    1: "EXC1 Doosan DX300LCA",
    2: "EXC2 Komatsu PC200",
    3: "DT1 HINO FM 280 JD",
    4: "DT2 Hino 130 HD",
    5: "BD1 Komatsu D85E",
    6: "BD2 Komatsu D65P",
    7: "VR1 XCMG XS113E",
    8: "VR2 Sakai SV512",
}

biaya = {
    1: 678569.75, 2: 501209.75,
    3: 584812.00, 4: 289156.00,
    5: 399993.75, 6: 370713.75,
    7: 438833.75, 8: 435025.75,
}

produkt = {
    1: 105.983, 2: 84.748,
    3: 5.867,   4: 2.070,
    5: 134.637, 6: 62.033,
    7: 124.600, 8: 120.000,
}

co2_per_jam = {
    1: 47.55, 2: 38.32,
    3: 76.08, 4: 52.76,
    5: 57.06, 6: 35.55,
    7: 38.04, 8: 33.60,
}

In [11]:
waktu_max      = 592
volume_bank    = 119422.82
volume_loose   = 204174.50
volume_compact = 128411.63
T              = 8
total_days     = 74

max_unit = {1:100, 2:100, 3:100, 4:100, 5:100, 6:100, 7:100, 8:100}

existing_cost_per_hari = 329_266_472.00
existing_co2_per_hari  = 41_083.20

alat = list(range(1, 9))

In [12]:
def solve_ceoe(mb_lo, mb_hi, skema="multitipe"):
    model = lp.LpProblem(f"CEOE_Mb{mb_lo}-{mb_hi}", lp.LpMinimize)

    x = {i: lp.LpVariable(f"x_{i}", lowBound=0, cat="Integer")
         for i in alat}

    model += lp.lpSum(biaya[i] * x[i] for i in alat)

    model += produkt[1]*x[1] + produkt[2]*x[2] >= volume_bank    / waktu_max
    model += produkt[3]*x[3] + produkt[4]*x[4] >= volume_loose   / waktu_max
    model += produkt[5]*x[5] + produkt[6]*x[6] >= volume_compact / waktu_max
    model += produkt[7]*x[7] + produkt[8]*x[8] >= volume_compact / waktu_max

    exc = produkt[1]*x[1] + produkt[2]*x[2]
    dtr = produkt[3]*x[3] + produkt[4]*x[4]
    model += exc >= mb_lo * dtr
    model += exc <= mb_hi * dtr

    model += produkt[5]*x[5] + produkt[6]*x[6] >= 0.25 * dtr

    for i in alat:
        model += x[i] <= max_unit[i]

    if skema == "single":
        for i in [2, 4, 6, 8]:
            model += x[i] == 0

    status = model.solve(lp.PULP_CBC_CMD(msg=0))

    if lp.LpStatus[status] != "Optimal":
        return None

    sol = {i: int(x[i].value()) for i in alat}
    cost_per_hari = sum(biaya[i] * sol[i] for i in alat) * T
    co2_per_hari  = sum(co2_per_jam[i] * sol[i] * T for i in alat)

    return {
        "mb_lo": mb_lo, "mb_hi": mb_hi,
        "solution": sol,
        "cost_per_hari": cost_per_hari,
        "co2_per_hari": co2_per_hari,
        "cost_saving_pct": (existing_cost_per_hari - cost_per_hari)
                           / existing_cost_per_hari * 100,
        "co2_saving_pct": (existing_co2_per_hari - co2_per_hari)
                          / existing_co2_per_hari * 100,
    }

In [13]:
mb_intervals = [
    (1.00, 1.00),
    (0.95, 1.05),
    (0.90, 1.10),
    (0.85, 1.20),
    (0.80, 1.25),
    (0.75, 1.25),
    (0.70, 1.30),
]

print("=" * 70)
print(f"{'Skema':<10} {'Mb Interval':<16} {'Biaya/Hari':>18} "
      f"{'Saving':>8} {'CO2/Hari':>12} {'Red.CO2':>8}")
print("=" * 70)

results = []
for skema in ["single", "multitipe"]:
    print(f"\n--- {skema.upper()} ---")
    for mb_lo, mb_hi in mb_intervals:
        r = solve_ceoe(mb_lo, mb_hi, skema)
        if r:
            results.append({**r, "skema": skema})
            print(f"Mb {mb_lo:.2f}-{mb_hi:.2f}      "
                  f"Rp {r['cost_per_hari']:>16,.0f}  "
                  f"{r['cost_saving_pct']:>6.2f}%  "
                  f"{r['co2_per_hari']:>10,.0f}  "
                  f"{r['co2_saving_pct']:>6.2f}%")
        else:
            print(f"Mb {mb_lo:.2f}-{mb_hi:.2f}      INFEASIBLE")

Skema      Mb Interval              Biaya/Hari   Saving     CO2/Hari  Red.CO2

--- SINGLE ---
Mb 1.00-1.00      INFEASIBLE
Mb 0.95-1.05      Rp      357,951,696   -8.71%      45,039   -9.63%
Mb 0.90-1.10      Rp      305,738,178    7.15%      38,573    6.11%
Mb 0.85-1.20      Rp      305,738,178    7.15%      38,573    6.11%
Mb 0.80-1.25      Rp      305,738,178    7.15%      38,573    6.11%
Mb 0.75-1.25      Rp      305,738,178    7.15%      38,573    6.11%
Mb 0.70-1.30      Rp      305,738,178    7.15%      38,573    6.11%

--- MULTITIPE ---
Mb 1.00-1.00      INFEASIBLE
Mb 0.95-1.05      Rp      305,430,288    7.24%      38,587    6.08%
Mb 0.90-1.10      Rp      305,430,288    7.24%      38,587    6.08%
Mb 0.85-1.20      Rp      304,258,370    7.60%      38,428    6.46%
Mb 0.80-1.25      Rp      304,258,370    7.60%      38,428    6.46%
Mb 0.75-1.25      Rp      302,839,490    8.03%      38,354    6.64%
Mb 0.70-1.30      Rp      301,420,610    8.46%      38,280    6.82%


In [14]:
# Hitung Mb aktual
def hitung_mb_aktual(solution):
    prod_exc = (produkt[1]*solution[1] + 
                produkt[2]*solution[2])
    prod_dtr = (produkt[3]*solution[3] + 
                produkt[4]*solution[4])
    if prod_dtr == 0:
        return None
    return prod_exc / prod_dtr


# Fungsi pilih rekomendasi
def pilih_rekomendasi(results, skema, haul_distance=None):
    skema_results = [r for r in results 
                     if r["skema"] == skema]
    
    for r in skema_results:
        r["mb_aktual"] = hitung_mb_aktual(r["solution"])
    
    if skema == "single":
        best = min(skema_results,
                   key=lambda r: abs(r["mb_aktual"] - 1.0))
    else:
        candidates = []
        for r in skema_results:
            sol = r["solution"]
            tipe_terpakai = sum(1 for i in [1,2] if sol[i] > 0)
            is_multitipe = tipe_terpakai > 1
            if is_multitipe:
                candidates.append(r)
        
        if not candidates:
            candidates = skema_results
        
        best = min(candidates,
                   key=lambda r: abs(r["mb_aktual"] - 1.0))
    
    return best


# Print hasil rekomendasi
def mb_label(mb):
    diff = abs(mb - 1.0)
    if diff < 0.10:
        return "optimal — sistem seimbang"
    elif diff < 0.20:
        return "mendekati 1.0 — adaptif di lapangan"
    else:
        return "jauh dari 1.0 — perlu evaluasi"

# Mb eksisting sebagai baseline
mb_eksisting = (produkt[1]*4) / (produkt[3]*60)

print("=" * 65)
print("CONSTRUCTION EQUIPMENT OPTIMIZATION ENGINE (CEOE)")
print("=" * 65)
print(f"\n{'--- KONDISI EKSISTING ---':^65}")
print(f"  {'EXC1 Doosan DX300LCA':<30}: 4 unit")
print(f"  {'DT1 HINO FM 280 JD':<30}: 60 unit")
print(f"  {'BD1 Komatsu D85E':<30}: 4 unit")
print(f"  {'VR1 XCMG XS113E':<30}: 4 unit")
print()
print(f"  Biaya/hari   : Rp {existing_cost_per_hari:,.0f}")
print(f"  CO2/hari     : {existing_co2_per_hari:,.2f} kgCO2")
print(f"  Mb aktual    : {mb_eksisting:.4f} → excavator idle")

for skema in ["single", "multitipe"]:
    r = pilih_rekomendasi(results, skema)
    
    print(f"\n{'='*65}")
    print(f"  SKEMA {skema.upper()}")
    print(f"{'='*65}")
    print(f"  Interval Mb  : {r['mb_lo']:.2f} - {r['mb_hi']:.2f}")
    print(f"  Mb aktual    : {r['mb_aktual']:.4f} "
          f"→ {mb_label(r['mb_aktual'])}")
    print()
    print(f"  Konfigurasi alat:")
    for i in alat:
        if r["solution"][i] > 0:
            print(f"    {nama_alat[i]:<30}: {r['solution'][i]} unit")
    print()
    print(f"  Biaya/hari        : Rp {r['cost_per_hari']:,.0f}")
    print(f"  CO2/hari          : {r['co2_per_hari']:,.2f} kgCO2")
    print(f"  Penghematan biaya : {r['cost_saving_pct']:.2f}% "
          f"(Rp {existing_cost_per_hari - r['cost_per_hari']:,.0f}/hari)")
    print(f"  Reduksi emisi     : {r['co2_saving_pct']:.2f}% "
          f"({existing_co2_per_hari - r['co2_per_hari']:,.2f} kgCO2/hari)")

# Comparison table
r_single = pilih_rekomendasi(results, "single")
r_multi  = pilih_rekomendasi(results, "multitipe")

print(f"\n{'='*65}")
print(f"  PERBANDINGAN DUA SKEMA")
print(f"{'='*65}")
print(f"  {'Aspek':<25} {'Tipe Tunggal':>16} {'Multitipe':>16}")
print(f"  {'-'*57}")
print(f"  {'Biaya/hari':<25} "
      f"Rp {r_single['cost_per_hari']:>10,.0f}  "
      f"Rp {r_multi['cost_per_hari']:>10,.0f}")
print(f"  {'Penghematan biaya':<25} "
      f"{r_single['cost_saving_pct']:>15.2f}%  "
      f"{r_multi['cost_saving_pct']:>15.2f}%")
print(f"  {'CO2/hari (kgCO2)':<25} "
      f"{r_single['co2_per_hari']:>16,.0f}  "
      f"{r_multi['co2_per_hari']:>16,.0f}")
print(f"  {'Reduksi emisi':<25} "
      f"{r_single['co2_saving_pct']:>15.2f}%  "
      f"{r_multi['co2_saving_pct']:>15.2f}%")
print(f"  {'Mb aktual':<25} "
      f"{r_single['mb_aktual']:>16.4f}  "
      f"{r_multi['mb_aktual']:>16.4f}")
print(f"  {'Fleksibilitas alat':<25} "
      f"{'Rendah':>16}  {'Tinggi':>16}")
print(f"  {'Risiko implementasi':<25} "
      f"{'Rendah':>16}  {'Sedang':>16}")
print(f"\n{'='*65}")
print(f"  REKOMENDASI: Skema Multitipe")
print(f"  Mb = {r_multi['mb_aktual']:.4f} — paling efisien dari")
print(f"  aspek biaya maupun emisi karbon.")
print(f"{'='*65}")

CONSTRUCTION EQUIPMENT OPTIMIZATION ENGINE (CEOE)

                    --- KONDISI EKSISTING ---                    
  EXC1 Doosan DX300LCA          : 4 unit
  DT1 HINO FM 280 JD            : 60 unit
  BD1 Komatsu D85E              : 4 unit
  VR1 XCMG XS113E               : 4 unit

  Biaya/hari   : Rp 329,266,472
  CO2/hari     : 41,083.20 kgCO2
  Mb aktual    : 1.2043 → excavator idle

  SKEMA SINGLE
  Interval Mb  : 0.95 - 1.05
  Mb aktual    : 1.0472 → optimal — sistem seimbang

  Konfigurasi alat:
    EXC1 Doosan DX300LCA          : 4 unit
    DT1 HINO FM 280 JD            : 69 unit
    BD1 Komatsu D85E              : 2 unit
    VR1 XCMG XS113E               : 2 unit

  Biaya/hari        : Rp 357,951,696
  CO2/hari          : 45,039.36 kgCO2
  Penghematan biaya : -8.71% (Rp -28,685,224/hari)
  Reduksi emisi     : -9.63% (-3,956.16 kgCO2/hari)

  SKEMA MULTITIPE
  Interval Mb  : 0.85 - 1.20
  Mb aktual    : 0.8572 → mendekati 1.0 — adaptif di lapangan

  Konfigurasi alat:
    EXC1 D